In [0]:
# Widget parameters
dbutils.widgets.text("volume_directory", "/Volumes/usa/osint/rss/", "Volume Directory")
dbutils.widgets.text("catalog_name", "usa", "Catalog Name")
dbutils.widgets.text("schema_name", "osint", "Schema Name")

# Retrieve parameter values
volume_directory = dbutils.widgets.get("volume_directory")
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")

print(f"Volume Directory: {volume_directory}")
print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")

In [0]:
import xml.etree.ElementTree as ET
import os
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime
from email.utils import parsedate_to_datetime

# List all XML files in the volume directory
xml_files = [f for f in os.listdir(volume_directory) if f.endswith(".xml")]
print(f"Found {len(xml_files)} XML files in {volume_directory}")

# Parse each XML file and extract RSS items
rows = []

for xml_file in xml_files:
    filepath = os.path.join(volume_directory, xml_file)
    try:
        tree = ET.parse(filepath)
        root = tree.getroot()

        # Handle RSS 2.0 format
        channel = root.find("channel")
        if channel is None:
            # Try Atom format or skip
            print(f"  Skipping {xml_file} - not standard RSS 2.0 format")
            continue

        feed_title = channel.findtext("title", default="")
        feed_link = channel.findtext("link", default="")

        for item in channel.findall("item"):
            # Parse pubDate to timestamp
            pub_date_str = item.findtext("pubDate", default=None)
            pub_date = None
            if pub_date_str:
                try:
                    pub_date = parsedate_to_datetime(pub_date_str)
                except Exception:
                    pub_date = None

            rows.append(Row(
                source_file=xml_file,
                feed_title=feed_title,
                feed_link=feed_link,
                item_title=item.findtext("title", default=None),
                item_link=item.findtext("link", default=None),
                item_description=item.findtext("description", default=None),
                item_pub_date=pub_date,
                item_guid=item.findtext("guid", default=None),
                item_category=item.findtext("category", default=None),
                item_author=item.findtext("author", default=item.findtext("{http://purl.org/dc/elements/1.1/}creator", default=None)),
                ingested_at=datetime.now(),
            ))

        print(f"  ✓ Parsed {xml_file}: {len(channel.findall('item'))} items")
    except Exception as e:
        print(f"  ✗ Error parsing {xml_file}: {e}")

print(f"\nTotal records extracted: {len(rows)}")

In [0]:
# Create DataFrame and write to bronze_rss table
schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("feed_title", StringType(), True),
    StructField("feed_link", StringType(), True),
    StructField("item_title", StringType(), True),
    StructField("item_link", StringType(), True),
    StructField("item_description", StringType(), True),
    StructField("item_pub_date", TimestampType(), True),
    StructField("item_guid", StringType(), True),
    StructField("item_category", StringType(), True),
    StructField("item_author", StringType(), True),
    StructField("ingested_at", TimestampType(), True),
])

df = spark.createDataFrame(rows, schema=schema)

# Write to the bronze_rss table (overwrite to refresh with latest feed data)
table_name = f"{catalog_name}.{schema_name}.bronze_rss"
df.write.mode("overwrite").saveAsTable(table_name)

print(f"✓ Written {df.count()} records to {table_name}")
display(df)